# dots.ocr — Colab vLLM Server

Notebook này khởi động vLLM server trên Colab GPU (T4 miễn phí),
sau đó tạo public URL qua ngrok để kết nối với UI local của bạn.

**Các bước:**
1. Đổi Runtime → GPU (T4)
2. Chạy từng cell theo thứ tự
3. Copy ngrok URL từ Cell 5 → dán vào ô **Server URL** trên UI local


## Cell 1 — Cài đặt dependencies

In [ ]:
# Kiểm tra GPU
!nvidia-smi

# Cài PyTorch và dependencies
!pip install torch==2.7.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 -q

# Clone repo
!git clone https://github.com/hoanggiangppe-tech/dots.ocr.git
%cd dots.ocr
!git checkout claude/setup-local-repo-4LmIb

# Cài package
!pip install -e . -q
print('\n✅ Dependencies installed')

## Cell 2 — Cài vLLM

In [ ]:
!pip install vllm==0.9.1 -q
print('✅ vLLM installed')

## Cell 3 — Tải model weights (~4GB, mất vài phút)

In [ ]:
!python3 tools/download_model.py
print('✅ Model downloaded')

## Cell 4 — Cài ngrok và lấy authtoken

In [ ]:
!pip install pyngrok -q

# ⚠️ Đăng ký miễn phí tại https://dashboard.ngrok.com/get-started/your-authtoken
# Sau đó dán authtoken vào đây:
NGROK_TOKEN = ""  # <-- dán token vào đây

if not NGROK_TOKEN:
    print('⚠️  Chưa có token!')
    print('   1. Truy cập: https://dashboard.ngrok.com/get-started/your-authtoken')
    print('   2. Copy token và dán vào NGROK_TOKEN = "..." ở trên')
    print('   3. Chạy lại cell này')
else:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    print('✅ ngrok token configured')

## Cell 5 — Tạo public URL và khởi động vLLM server

In [ ]:
import subprocess, time, os
from pyngrok import ngrok

# Patch vllm để load custom model code
import shutil
vllm_bin = shutil.which('vllm')
with open(vllm_bin, 'r') as f:
    content = f.read()
patch_line = 'from DotsOCR import modeling_dots_ocr_vllm'
trigger = 'from vllm.entrypoints.cli.main import main'
if patch_line not in content:
    content = content.replace(trigger, trigger + '\n' + patch_line)
    with open(vllm_bin, 'w') as f:
        f.write(content)
    print('✅ vllm patched')
else:
    print('✅ vllm already patched')

# Tạo ngrok tunnel
tunnel = ngrok.connect(8000, bind_tls=True)
public_url = tunnel.public_url
print('\n' + '='*50)
print(f'🌐 PUBLIC URL: {public_url}')
print('='*50)
print('\n👆 Copy URL này và dán vào ô Server URL trong UI local của bạn!\n')

# Lưu URL ra file để tiện copy
with open('server_url.txt', 'w') as f:
    f.write(public_url)

# Khởi động vLLM server (chạy nền)
os.environ['PYTHONPATH'] = f"{os.path.dirname('./weights/DotsOCR')}:{os.environ.get('PYTHONPATH', '')}"
cmd = [
    'vllm', 'serve', './weights/DotsOCR',
    '--tensor-parallel-size', '1',
    '--gpu-memory-utilization', '0.9',
    '--chat-template-content-format', 'string',
    '--served-model-name', 'model',
    '--trust-remote-code',
    '--port', '8000',
]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Chờ server sẵn sàng
print('⏳ Khởi động vLLM server (30-60 giây)...')
import requests
for i in range(120):
    time.sleep(3)
    try:
        r = requests.get('http://localhost:8000/v1/models', timeout=2)
        if r.status_code == 200:
            print(f'\n✅ Server sẵn sàng sau {(i+1)*3} giây!')
            print(f'\n🔗 Dán URL này vào UI: {public_url}')
            break
    except:
        print(f'   [{(i+1)*3}s] Đang khởi động...')
else:
    print('⚠️ Server khởi động lâu hơn dự kiến. Kiểm tra logs bên dưới.')


## Cell 6 — Giữ server sống (chạy khi cần)
Chạy cell này để xem logs và giữ server không bị timeout.

In [ ]:
# Đọc logs từ vLLM server
for line in proc.stdout:
    print(line, end='')